# A Strands agent on Bedrock with Notion, through the Highflame AI gateway and MCP gateway

An [AWS Strands](https://strandsagents.com) agent on an Amazon Bedrock model, with the tools of
[Notion's hosted MCP server](https://developers.notion.com/docs/mcp). Here **all** of the agent's
traffic goes through Highflame: its model calls through the AI gateway, and its MCP calls through
the MCP gateway.

| Traffic | Path |
| --- | --- |
| Model calls | agent → **AI gateway** → Bedrock |
| Notion sign-in (OAuth) | you, once, in Studio ↔ Notion. Highflame stores the token for your user. |
| MCP calls (`tools/list`, `tools/call`) | agent → **MCP gateway** → `mcp.notion.com` |

**The agent holds no Notion credential.** It sends only your Highflame API key. The MCP gateway
finds the Notion token stored for the Studio user that key belongs to, adds it to the request to
Notion, and never returns it.

**What the MCP gateway adds.** Only the tools you enable in Studio are listed or callable. Each
tool call and each tool result is checked by your guardrail policies, and each MCP request is
recorded, so a tool call the agent makes on its own is covered too, not only those the model asks
for.

The same agent with the MCP traffic going directly to Notion is
[`strands_bedrock_mcp_via_ai_gateway.ipynb`](strands_bedrock_mcp_via_ai_gateway.ipynb).

## Setup

### 1. Add Bedrock as a provider on the AI gateway

**Studio → AI Gateway → LLM Providers → AWS Bedrock.** Set **Bedrock Endpoint** to the runtime
endpoint of the region you will call, for example `https://bedrock-runtime.us-east-1.amazonaws.com`.

The gateway holds no AWS credential and signs nothing. It forwards a **Bedrock API key** to Bedrock
as a bearer token, and the setup cell makes one for you: a short-term key minted from the AWS
credentials you already use (`AWS_PROFILE`, SSO, environment variables). The AWS identity needs
`bedrock:CallWithBearerToken` and permission to invoke the model. Prefer a long-term key from the
Bedrock console? Put it in `BEDROCK_API_KEY` and nothing is minted.

A model is named `bedrock/<model id>`, such as
`bedrock/us.anthropic.claude-sonnet-4-5-20250929-v1:0`. The gateway serves Bedrock on its
OpenAI-compatible endpoint, so the agent uses Strands' `OpenAIModel` pointed at the gateway, not
`BedrockModel`, which calls AWS directly.

### 2. Register Notion in the MCP gateway

**Studio → AI Gateway → MCP Servers → Register MCP Server.**

| Field | Value |
| --- | --- |
| Name | `notion`. It becomes the last part of the server's gateway URL, `https://gateway.highflame.ai/mcp/notion`. |
| Server URL | `https://mcp.notion.com/mcp` |
| Authentication | **OAuth Broker**. Leave the OAuth provider settings empty: with no client ID, Studio registers Highflame with Notion by itself. |

### 3. Connect your Notion account

Open the server in Studio and click **Connect with OAuth**. Sign in to Notion and approve access to
the pages you choose. Highflame stores the token for **your Studio user**.

Create the API key for this notebook as that same user. The gateway finds the Notion token by the
user the key belongs to, so a key created by someone else gets `403 downstream_auth_required`.

### 4. Enable the tools

Tools are disabled until you enable them. On the server's **Tools** tab, enable at least
`notion-search` and `notion-fetch`. Whatever you enable, the agent gets only the read-only tools:
the notebook filters out the rest.

### 5. The `.env` file

Copy `.env.example` to `.env` beside this notebook.

| Variable | What it is |
| --- | --- |
| `HIGHFLAME_API_KEY` | **Required.** Studio → AI Gateway → Settings → API Keys, created by the user from step 3. |
| `HIGHFLAME_GATEWAY_BASE_URL` | **Required.** `https://gateway.highflame.ai/llm/v1`, or your deployment's LLM base URL with `/v1`. |
| `HIGHFLAME_MCP_URL` | Optional. Defaults to `https://gateway.highflame.ai/mcp/notion`, the gateway URL for the name in step 2. The longer URL Studio shows, with your account and project in it, works too. |
| `AWS_REGION` | Optional. The region of the Bedrock endpoint from step 1. Unset, your AWS profile's region is used. |
| `AWS_PROFILE` | Optional. The AWS profile to mint the Bedrock API key from. |
| `BEDROCK_MODEL_ID` | Optional. Defaults to `us.anthropic.claude-sonnet-4-5-20250929-v1:0`. |
| `BEDROCK_API_KEY` | Optional. A Bedrock API key to use instead of minting one. |

Note: You can use this notebook for other providers too.

In [ ]:
%pip install -q -r requirements.txt

from importlib.metadata import version

for package in ("strands-agents", "mcp", "highflame"):
    print(package, version(package))

In [ ]:
import asyncio
import collections
import os

from dotenv import load_dotenv
from openai import OpenAI, OpenAIError
from strands import Agent
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient

load_dotenv() 

HIGHFLAME_API_KEY = os.environ["HIGHFLAME_API_KEY"]
GATEWAY_BASE_URL = os.environ["HIGHFLAME_GATEWAY_BASE_URL"]
AWS_REGION = os.environ.get("AWS_REGION")
BEDROCK_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID") or "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
MODEL = f"bedrock/{BEDROCK_MODEL_ID}"

QUESTION = "Search my Notion workspace and list the titles of up to three pages you can find, with one line about each."

# The gateway forwards this to Bedrock as `Authorization: Bearer`. A minted key lasts up to 12
# hours, and never longer than the AWS session it came from.
BEDROCK_API_KEY = os.environ.get("BEDROCK_API_KEY")
if not BEDROCK_API_KEY:
    from aws_bedrock_token_generator import provide_token

    BEDROCK_API_KEY = provide_token(region=AWS_REGION)


def gateway_model() -> OpenAIModel:
    """Bedrock, reached through the Highflame AI gateway.

    `api_key` is sent as `Authorization: Bearer` and forwarded to Bedrock. The Highflame key travels
    in its own header, so the two credentials never compete for one."""
    return OpenAIModel(
        client_args={
            "base_url": GATEWAY_BASE_URL,
            "api_key": BEDROCK_API_KEY,
            "default_headers": {"X-Highflame-APIKey": HIGHFLAME_API_KEY},
        },
        model_id=MODEL,
        params={"max_tokens": 1024, "temperature": 0},
    )

REFUSAL_MARKERS = ("Highflame Security:", "was blocked by security policy")


def show(result) -> None:
    """Print the agent's answer, whether Highflame refused a step, and which tools ran."""
    text = str(result).strip()
    print(("Refused by Highflame: " if any(m in text for m in REFUSAL_MARKERS) else "") + text)
    print("\ntools the agent ran:", ", ".join(result.metrics.tool_metrics) or "none")


try:
    probe = OpenAI(
        base_url=GATEWAY_BASE_URL, api_key=BEDROCK_API_KEY, default_headers={"X-Highflame-APIKey": HIGHFLAME_API_KEY}
    ).chat.completions.create(
        model=MODEL, max_tokens=5, messages=[{"role": "user", "content": "Reply with the single word: ok"}]
    )
except OpenAIError as exc:
    raise RuntimeError(
        f"The gateway did not answer a Bedrock call: {exc}\n"
        "Check HIGHFLAME_GATEWAY_BASE_URL and HIGHFLAME_API_KEY, that AWS Bedrock is added under "
        "Studio -> AI Gateway -> LLM Providers for AWS_REGION, and that your AWS identity may call the model."
    ) from exc
if probe.id.startswith("chatcmpl-blocked-"):
    raise RuntimeError(f"The gateway refused a plain prompt: {probe.choices[0].message.content!r}")

print("gateway:", GATEWAY_BASE_URL)
print("model  :", MODEL, "in", AWS_REGION)
print("probe  :", probe.choices[0].message.content)

## 1. The model, through the gateway

A plain turn first, with no tools. The model call leaves this process for the gateway, which
checks the prompt, records the call against your API key and forwards it to Bedrock. The reply
comes back the same way.

In [ ]:
agent = Agent(model=gateway_model(), callback_handler=None)
show(await agent.invoke_async("In one sentence, what is the Model Context Protocol?"))

## 2. Connect to Notion, through the MCP gateway

The agent never signs in to Notion. You did that once, in Studio, and the MCP gateway holds the
token:

1. The MCP client sends each request to the gateway with one header, your Highflame API key.
2. The gateway finds the Notion token stored for the Studio user that key belongs to.
3. It adds that token to the request to Notion, and returns Notion's answer without it.

No Notion token enters this process, and there is no OAuth code in this notebook.

The cell first sends one `initialize` request by hand, so a setup gap is named here instead of
surfacing as a generic connection error from inside the agent:

| Answer | Meaning |
| --- | --- |
| `200` | Ready. |
| `401 unauthorized` | No server is registered at `HIGHFLAME_MCP_URL`, or the API key is wrong. The gateway answers an unknown server as unauthenticated, so check the name from setup step 2 first. |
| `403 downstream_auth_required` | Notion is registered, but no Notion account is connected for the user this key belongs to. Do setup step 3 as that user. |

Every request the MCP client sends is also written to `MCP_TRAFFIC`, with the names of the
credential headers it carried, so the next cell can show where each one went. An httpx auth hook is
the one place Strands exposes each outgoing request, so the recorder is one. It adds no credential.

The agent gets Notion's **read-only** tools only. Nothing here writes to your workspace.

In [ ]:
import httpx

MCP_SERVER_URL = os.environ.get("HIGHFLAME_MCP_URL") or "https://gateway.highflame.ai/mcp/notion"
READ_ONLY_TOOLS = [
    "notion-search",
    "notion-fetch",
    "notion-get-users",
    "notion-get-teams",
    "notion-get-comments"
]
CREDENTIAL_HEADERS = ("authorization", "x-highflame-apikey", "x-highflame-token")
MCP_TRAFFIC: list[tuple[str, str, str, tuple[str, ...]]] = []  # (method, host, path, credential headers) of every request the MCP client sent

# One initialize by hand, so a setup gap is named here and not inside the agent.
answer = httpx.post(
    MCP_SERVER_URL,
    json={
        "jsonrpc": "2.0",
        "id": 1,
        "method": "initialize",
        "params": {"protocolVersion": "2025-11-25", "capabilities": {}, "clientInfo": {"name": "preflight", "version": "0"}},
    },
    headers={"X-Highflame-APIKey": HIGHFLAME_API_KEY, "Accept": "application/json, text/event-stream"},
    timeout=60,
)
if answer.status_code != 200:
    try:
        detail = answer.json()
    except ValueError:
        detail = {"error": answer.text[:200]}
    hints = {
        401: "No server at HIGHFLAME_MCP_URL, or a wrong API key. Check the server name (setup step 2).",
        403: "No Notion account connected for this key's user. Do setup step 3 as that user.",
    }
    raise RuntimeError(
        f"MCP gateway answered {answer.status_code} {detail.get('error', '')}: {detail.get('error_description', '')}\n"
        + hints.get(answer.status_code, "See the answer above.")
    )


class RequestLog(httpx.Auth):
    """Records each request the MCP client sends. Not authentication: it adds nothing."""

    def auth_flow(self, request):
        sent = tuple(sorted(name for name in (h.lower() for h in request.headers) if name in CREDENTIAL_HEADERS))
        MCP_TRAFFIC.append((request.method, request.url.host, request.url.path, sent))
        yield request


notion = MCPClient(
    url=MCP_SERVER_URL,
    headers={"X-Highflame-APIKey": HIGHFLAME_API_KEY},
    auth_provider=RequestLog(),
    tool_filters={"allowed": READ_ONLY_TOOLS},
)
notion.start()
notion_tools = notion.list_tools_sync()
if not notion_tools:  # with no tool enabled, the gateway lists only a placeholder, which the filter drops
    notion.stop(None, None, None)
    raise RuntimeError("No read-only Notion tool is enabled for this server. Enable notion-search and notion-fetch (setup step 4).")
print("connected to", MCP_SERVER_URL)
print("read-only tools the agent gets:", ", ".join(t.tool_name for t in notion_tools))

In [ ]:
def summarize_traffic() -> None:
    """Where each request the MCP client sent went, and which credential headers it carried."""
    counts = collections.Counter((host, path, sent) for _, host, path, sent in MCP_TRAFFIC)
    for (host, path, sent), n in counts.items():
        print(f"  {host:24} {path:28} {', '.join(sent) or 'no credential':24} {n}")


print("Requests the MCP client sent so far:")
summarize_traffic()
print("\nNone went to Notion, and none carried a Notion token.")

## 3. The agent: all traffic through Highflame

When the model asks for a Notion tool, the AI gateway checks the call on its way back to the agent.
The agent then calls the tool through the MCP gateway, which checks it again against this server's
enabled tools and your policies, adds your Notion token and forwards it. The result comes back
through the MCP gateway, and goes to the model through the AI gateway.

In [ ]:
notion_agent = Agent(
    model=gateway_model(),
    tools=notion_tools,
    system_prompt="You help the user find things in their Notion workspace. Use the Notion tools to answer.",
    callback_handler=None,  # print the answer once, below, instead of streaming it
)
show(await notion_agent.invoke_async(QUESTION))

## Clean up

This stops the MCP connection.

The Notion server and your connection to it stay in Studio for the next run. To remove them,
delete the server under **AI Gateway → MCP Servers**, and disconnect Highflame in Notion's
**Settings → Connections**.

In [ ]:
notion.stop(None, None, None)
print("stopped")